# Separate pressure-difference auxiliary head — V019

Prepared next mechanism; no approved-data fit has run. Shared encoder, original pressure head, separate first-difference output. Inspired by the user-supplied discussion synthesis, with provenance in DISCUSSION_INSPIRED_ROADMAP_v015.md. Training pressure differences are auxiliary targets only; inference receives known controls only. Validation and checkpoint selection remain native inspiratory mean MAE.

Depends on reviewed feature/sequence/training V005/V008 notebooks being loaded first. Fresh private V008 engines supply checkpoint, optimizer, RNG, scheduler and best-validation selection machinery. Checkpoints use the V8 container with an additional mandatory v019 loss/source/weight identity. The base v008 adapter is l1; v019 records the actual multi-task objective. No training, data reads, allocation or submission happens by defining this notebook.


In [ ]:
import copy, json, hashlib, math
from pathlib import Path
import numpy as np
import torch
from torch import nn

V019_VERSION='v019-separate-first-difference-head-1'
V019_COMPONENT='auxiliary_difference_components_v019.ipynb'

class V019AuxiliaryPressureRNN(V5PressureRNN):
    """Pressure and a separate auxiliary difference output share an encoder.

    Only known input features enter the network. Adjacent pressure labels are
    auxiliary training targets, never inference inputs. Every breath starts with
    fresh recurrent state. The last auxiliary output is a diagnostic cache, not
    recurrent memory or an input to the next call.
    """
    def __init__(self,model_spec,n_features):
        super().__init__(model_spec,n_features)
        with torch.random.fork_rng(devices=[]):
            self.difference_head=nn.Sequential(nn.Linear(self.head[0].in_features,self.spec['head_width']),nn.SiLU(),nn.Linear(self.spec['head_width'],9 if self.spec['rc_heads'] else 1))
        self.difference_prediction_last=None
    def forward(self,x):
        if x.ndim!=3 or x.shape[-1]!=self.n_features+1:raise ValueError('Expected transformed complete input breaths')
        joint=x[...,-1].long();sequence=x[...,:self.n_features]
        if self.joint_embedding is not None:sequence=torch.cat((sequence,self.joint_embedding(joint)),dim=-1)
        if not self.explicit_stack:sequence,_=self.rnn(sequence)
        else:
            outputs=[]
            for i,layer in enumerate(self.recurrent_layers):
                previous=sequence;sequence,_=layer(previous)
                if self.spec['residual']:sequence=sequence+self.skip_projections[i](previous)
                outputs.append(sequence)
                if i+1<len(self.recurrent_layers):sequence=self.between_layers(sequence)
            if self.spec['output_concat']:sequence=torch.cat(outputs,dim=-1)
        prediction=self.head(sequence);difference=self.difference_head(sequence)
        if self.spec['rc_heads']:
            prediction=prediction.gather(-1,joint.unsqueeze(-1));difference=difference.gather(-1,joint.unsqueeze(-1))
        self.difference_prediction_last=difference.squeeze(-1)
        return prediction.squeeze(-1)

def v019_auxiliary_factory(spec,n_features,prep):return V019AuxiliaryPressureRNN(spec,n_features)
v019_auxiliary_factory.__module__='auxiliary_difference_components_v019'

def auxiliary_difference_loss_v019(prediction,difference_prediction,target,inspiration_mask,aux_weight=.1):
    if not np.isfinite(aux_weight) or aux_weight<0:raise ValueError('Auxiliary weight must be finite and nonnegative')
    if prediction.ndim!=2 or target.shape!=prediction.shape or difference_prediction.shape!=prediction.shape or inspiration_mask.shape!=prediction.shape:raise ValueError('Expected matching breath/step arrays')
    mask=inspiration_mask.bool()
    if not bool(mask.any()):raise ValueError('Need inspired points')
    main=(prediction-target).masked_select(mask).abs().mean();adjacent=mask[:,1:]&mask[:,:-1]
    difference_error=difference_prediction[:,1:]-(target[:,1:]-target[:,:-1])
    auxiliary=difference_error.masked_select(adjacent).abs().mean() if bool(adjacent.any()) else difference_prediction[:,:0].sum()
    # Weight-zero control has no auxiliary graph/optimizer updates.
    objective=main if aux_weight==0 else main+float(aux_weight)*auxiliary
    return objective,main,auxiliary

def _v019_metadata(aux_weight,source_fingerprints):
    if not np.isfinite(aux_weight) or aux_weight<0:raise ValueError('Invalid auxiliary weight')
    path=_v8_notebook_path(V019_COMPONENT);source=notebook_code_sha256_v008(path)
    if not source_fingerprints or source_fingerprints.get(V019_COMPONENT)!=source:raise ValueError('Exact auxiliary component source pin required')
    return dict(version=V019_VERSION,loss_kind='separate_first_difference_head_l1',aux_weight=float(aux_weight),source_code_sha256=source,source_fingerprints=dict(sorted(source_fingerprints.items())),base_checkpoint_container=8,base_adapter_loss_kind='l1',auxiliary_phase='adjacent inspiratory pairs only',auxiliary_target='scaled pressure first difference',validation_metric='native inspiration-only point-weighted mean absolute pressure error',inference_output='pressure head only')

def _v019_engine(arm,aux_weight,source_fingerprints,device):
    meta=_v019_metadata(aux_weight,source_fingerprints)
    engine,base_meta=_v8_engine(arm,'l1',v019_auxiliary_factory,source_fingerprints,device)
    old_fingerprint=engine['_fingerprint'];old_payload=engine['_checkpoint_payload'];old_prepare=engine['_prepare_run'];old_configuration=engine['_train_configuration'];latest={}
    def configuration(cfg):
        c=old_configuration(cfg)
        if c['residual_baseline']:raise ValueError('This first-difference experiment requires no varying target baseline')
        return c
    def fingerprint(*args,**kwargs):
        r=old_fingerprint(*args,**kwargs);r['v019']=copy.deepcopy(meta);return r
    def payload(model,prep,spec,cfg,seed,run_fingerprint,optimizer,scheduler,generator,epoch,history,completed=False,termination=None):
        # Mutate the engine's authoritative history before its payload deep copy,
        # so earlier epochs retain their auxiliary diagnostics on later saves.
        if history and history[-1]['epoch']==epoch:history[-1].update(copy.deepcopy(latest))
        r=old_payload(model,prep,spec,cfg,seed,run_fingerprint,optimizer,scheduler,generator,epoch,history,completed,termination)
        r['v019']=copy.deepcopy(meta);return r
    def prepare(raw,y,out,spec,cfg,seed,feature_names,checkpoint_path,resume,run_fingerprint,selected_device):
        if resume:
            saved=torch.load(engine['_last_path'](checkpoint_path),map_location='cpu',weights_only=False)
            if saved.get('v019')!=meta or saved.get('fingerprint',{}).get('v019')!=meta:raise ValueError('Auxiliary resume source/loss/weight changed')
        return old_prepare(raw,y,out,spec,cfg,seed,feature_names,checkpoint_path,resume,run_fingerprint,selected_device)
    def training_epoch(model,loader,optimizer,cfg,prep,selected_device):
        model.train();total_error=0.;total_count=0;steps=0;adj_error=0.;adj_count=0;objective_sum=0.
        for xb,yb,mask in loader:
            xb,yb,mask=(v.to(selected_device,non_blocking=selected_device.type=='cuda') for v in (xb,yb,mask))
            if not bool(mask.any()):continue
            optimizer.zero_grad(set_to_none=True);p=model(xb)
            loss,main,aux=auxiliary_difference_loss_v019(p,model.difference_prediction_last,yb,mask,aux_weight)
            if not torch.isfinite(loss):raise RuntimeError('Nonfinite auxiliary loss')
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),float(cfg['gradient_clip']));optimizer.step();steps+=1
            n=int(mask.sum());pairs=int((mask[:,1:]&mask[:,:-1]).sum())
            total_error+=float((p-yb).masked_select(mask).detach().abs().sum())*prep['target_scale'];total_count+=n
            adj_error+=float(aux.detach())*pairs*prep['target_scale'];adj_count+=pairs;objective_sum+=float(loss.detach())*n
        if not total_count:raise ValueError('No scored points')
        latest.clear();latest.update(training_loss_kind=meta['loss_kind'],training_auxiliary_weight=float(aux_weight),training_online_objective_scaled=objective_sum/total_count,training_adjacent_inspiratory_pairs=adj_count,training_online_auxiliary_difference_mae=adj_error/max(adj_count,1))
        return float(total_error/total_count),total_count,steps
    engine.update(_train_configuration=configuration,_fingerprint=fingerprint,_checkpoint_payload=payload,_prepare_run=prepare,_training_epoch=training_epoch)
    return engine,meta,base_meta

def fit_validate_v019(x_train,y_train,u_out_train,x_val,y_val,u_out_val,model_spec,train_cfg,seed,checkpoint_path,epoch_log_path=None,feature_names=None,train_diagnostic_breaths=2048,device='cuda',resume=False,on_epoch=None,*,aux_weight=.1,arm='v019_auxiliary',source_fingerprints=None):
    engine,meta,base=_v019_engine(arm,aux_weight,source_fingerprints,device)
    r=engine['fit_validate'](x_train,y_train,u_out_train,x_val,y_val,u_out_val,model_spec,train_cfg,seed,checkpoint_path,epoch_log_path=epoch_log_path,feature_names=feature_names,train_diagnostic_breaths=train_diagnostic_breaths,device=device,resume=resume,on_epoch=on_epoch)
    r=_v8_report(r,base);r.update(v019=meta,loss_kind=meta['loss_kind'],aux_weight=float(aux_weight));return r

def fit_fixed_v019(x_train,y_train,u_out_train,model_spec,train_cfg,seed,lr_schedule,checkpoint_path,epoch_log_path=None,feature_names=None,device='cuda',resume=False,on_epoch=None,*,aux_weight=.1,arm='v019_auxiliary',source_fingerprints=None):
    engine,meta,base=_v019_engine(arm,aux_weight,source_fingerprints,device)
    r=engine['fit_fixed'](x_train,y_train,u_out_train,model_spec,train_cfg,seed,lr_schedule,checkpoint_path,epoch_log_path=epoch_log_path,feature_names=feature_names,device=device,resume=resume,on_epoch=on_epoch)
    r=_v8_report(r,base);r.update(v019=meta,loss_kind=meta['loss_kind'],aux_weight=float(aux_weight));return r

def load_checkpoint_v019(path,device='cpu',*,aux_weight=.1,expected_arm='v019_auxiliary',source_fingerprints=None):
    saved=torch.load(path,map_location='cpu',weights_only=False);meta=_v019_metadata(aux_weight,source_fingerprints)
    if saved.get('v019')!=meta or saved.get('fingerprint',{}).get('v019')!=meta:raise ValueError('Auxiliary checkpoint source/loss/weight differs')
    return load_checkpoint_v008(path,device=device,model_factory=v019_auxiliary_factory,expected_arm=expected_arm,expected_loss_kind='l1',source_fingerprints=source_fingerprints)

def predict_checkpoint_v019(path,x,device='cpu',batch_size=512,*,aux_weight=.1,expected_arm='v019_auxiliary',source_fingerprints=None):
    if int(batch_size)<1:raise ValueError('Positive prediction batch size required')
    model,prep,saved=load_checkpoint_v019(path,device,aux_weight=aux_weight,expected_arm=expected_arm,source_fingerprints=source_fingerprints)
    engine,_,_=_v019_engine(expected_arm,aux_weight,source_fingerprints,device)
    return engine['_predict_native'](model,engine['transform_inputs'](x,prep),prep,engine['_sequence_device'](device),int(batch_size))
